In [1]:
from main import *
import folium
import polyline as pl
from tqdm import tqdm
import random

### Data preparation
Extract polylines for each trip using the googlemaps routes api

In [2]:
df = pd.read_feather(BIXI_FEATHER_PATH)
df = df.dropna(subset=START_COLS+END_COLS)
df["directions"] = None

requestCounter = 0
MAX_ID = len(df)

# create random range
RANGE_SIZE = 1000
rng = np.random.default_rng(42) # seed
idx_range = rng.choice(np.arange(1, MAX_ID), size=RANGE_SIZE, replace=False)

try:
    with open("cachedDirections.pkl", "rb") as file:
        cachedDirections: dict[tuple[Point2D, Point2D], Polyline] = pickle.load(file)
except FileNotFoundError:
    cachedDirections: dict[tuple[Point2D, Point2D], Polyline] = dict()

routes_list = []
for idx in tqdm(idx_range):
    trip = df.iloc[idx]
    tripKey=tuple(trip[START_COLS+END_COLS].values)
    directions = None
    if tripKey in cachedDirections:
        directions = cachedDirections[tripKey]
    else:
        # print(trip[START_COLS].values)
        directions = get_route(tuple(trip[START_COLS].values), tuple(trip[END_COLS].values))
        cachedDirections[tripKey] = directions
        requestCounter += 1
    routes_list.append(directions)
df.iloc[idx_range, df.columns.get_loc("directions")] = routes_list

with open("cachedDirections.pkl", "wb") as file:
    pickle.dump(cachedDirections, file)

print(requestCounter, "requests /", MAX_ID, f" polylines extracted     ({requestCounter/MAX_ID*100}%)")

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [01:31<00:00, 10.92it/s]

989 requests / 14197540  polylines extracted     (0.006965995517533319%)


### Map visualization
Generate heatmap of most frequently used roads

In [3]:
# Create a map with folium
ROOT_COORDS = df[START_COLS].iloc[7000000].values
m = folium.Map(location=ROOT_COORDS, zoom_start=12, tiles="cartodb positron")
# Extract polylines and add to the map

for i, directions in enumerate(df.iloc[idx_range]["directions"]):
    encoded_polyline = directions.routes[0].polyline.encoded_polyline
    polyline = pl.decode(encoded_polyline)
    folium.PolyLine(
        locations=polyline,
        color="red",
        weight=3,
        opacity=0.05,
        tooltip=f"{i}",
    ).add_to(m)
m

In [50]:
idx_range[:5]

array([ 82, 877, 621, 511,  81])

In [33]:
df.iloc[0:].drop_duplicates(subset=START_COLS+END_COLS, keep='first')

,STARTSTATIONNAME,STARTSTATIONARRONDISSEMENT,STARTSTATIONLATITUDE,STARTSTATIONLONGITUDE,ENDSTATIONNAME,ENDSTATIONARRONDISSEMENT,ENDSTATIONLATITUDE,ENDSTATIONLONGITUDE,STARTTIMEMS,ENDTIMEMS,polyline
3440,St-Viateur / Casgrain,Le Plateau-Mont-Royal,45.527013,-73.597973,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767118845159,1.767120e+12,routes {\n distance_meters: 4456\n duration ...
3441,9e avenue / Masson,Rosemont - La Petite-Patrie,45.549490,-73.573272,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767146967447,1.767148e+12,routes {\n distance_meters: 1888\n duration ...
3442,St-Urbain / Laurier,Le Plateau-Mont-Royal,45.521711,-73.593743,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1766867149246,1.766868e+12,routes {\n distance_meters: 5586\n duration ...
3443,Parc de Turin (de Lanaudière / Jean-Talon),Villeray—Saint-Michel—Parc-Extension,45.545350,-73.610330,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1767227508567,1.767228e+12,routes {\n distance_meters: 2044\n duration ...
3444,Parthenais / du Mont-Royal,Le Plateau-Mont-Royal,45.536404,-73.571413,Parc Molson (d'Iberville / Beaubien),Rosemont - La Petite-Patrie,45.549022,-73.591982,1766953764324,1.766955e+12,routes {\n distance_meters: 2722\n duration ...
...,...,...,...,...,...,...,...,...,...,...,...
14248779,du Mont-Royal / St-André,Le Plateau-Mont-Royal,45.526533,-73.580528,Bennett / Ste-Catherine,Mercier - Hochelaga-Maisonneuve,45.553159,-73.532633,1764772375864,1.764774e+12,None
14248847,du Mont-Royal / Christophe-Colomb,Le Plateau-Mont-Royal,45.527954,-73.579362,Bennett / Ste-Catherine,Mercier - Hochelaga-Maisonneuve,45.553159,-73.532633,1765292782311,1.765294e+12,None
14248850,Métro Berri-UQAM (St-Denis / de Maisonneuve),Ville-Marie,45.514252,-73.561502,Bennett / Ste-Catherine,Mercier - Hochelaga-Maisonneuve,45.553159,-73.532633,1765841417437,1.765843e+12,None
14248928,de Lanaudière / Rosemont,Rosemont - La Petite-Patrie,45.537844,-73.592346,Bennett / Ste-Catherine,Mercier - Hochelaga-Maisonneuve,45.553159,-73.532633,1763841453008,1.763844e+12,None
